# Classifying Penguins with Keras

In [164]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [165]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [166]:
#drop Nan rows
penguins = penguins.dropna()
penguins.shape

(333, 8)

In [167]:
#shuffle the data
penguins = penguins.sample(frac=1, random_state=42).reset_index(drop=True)
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Dream,39.5,16.7,178.0,3250.0,female,2007
1,Chinstrap,Dream,50.9,17.9,196.0,3675.0,female,2009
2,Adelie,Torgersen,42.1,19.1,195.0,4000.0,male,2008
3,Gentoo,Biscoe,46.6,14.2,210.0,4850.0,female,2008
4,Adelie,Biscoe,41.1,18.2,192.0,4050.0,male,2008


In [168]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,3250.0,39.5,16.7,178.0,True,False
1,3675.0,50.9,17.9,196.0,True,False
2,4000.0,42.1,19.1,195.0,False,True
3,4850.0,46.6,14.2,210.0,True,False
4,4050.0,41.1,18.2,192.0,False,True
...,...,...,...,...,...,...
328,4750.0,49.6,15.0,216.0,False,True
329,3900.0,37.2,19.4,184.0,False,True
330,3200.0,39.7,17.7,193.0,True,False
331,3950.0,45.2,17.8,198.0,True,False


In [169]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.152778,0.269091,0.428571,0.101695,1.0,0.0
1,0.270833,0.683636,0.571429,0.406780,1.0,0.0
2,0.361111,0.363636,0.714286,0.389831,0.0,1.0
3,0.597222,0.527273,0.130952,0.644068,1.0,0.0
4,0.375000,0.327273,0.607143,0.338983,0.0,1.0
...,...,...,...,...,...,...
328,0.569444,0.636364,0.226190,0.745763,0.0,1.0
329,0.333333,0.185455,0.750000,0.203390,0.0,1.0
330,0.138889,0.276364,0.547619,0.355932,1.0,0.0
331,0.347222,0.476364,0.559524,0.440678,1.0,0.0


In [170]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: object


array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,

In [171]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [172]:
model.summary()

Model: "penguin_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 7)              │            49 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 3)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119 (476.00 B)

 Trainable params: 119 (476.00 B)

 Non-trainable params: 0 (0.00 B)

In [173]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [174]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs=100, validation_split=0.1)

scores = model.evaluate(scaled_penguins_x, penguins_y, verbose=2)

Epoch 1/100


/Users/pmazolew/Documents/GitHub/AdvancedMachineLearning/pycaret-env/lib/python3.10/site-packages/keras/src/backend/tensorflow/nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2241 - loss: 1.1119 - val_accuracy: 0.2353 - val_loss: 1.1131
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3010 - loss: 1.1010 - val_accuracy: 0.1765 - val_loss: 1.1071
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3946 - loss: 1.0951 - val_accuracy: 0.4412 - val_loss: 1.1021
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4348 - loss: 1.0912 - val_accuracy: 0.4412 - val_loss: 1.0986
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4849 - loss: 1.0881 - val_accuracy: 0.4118 - val_loss: 1.0954
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4883 - loss: 1.0856 - val_accuracy: 0.4118 - val_loss: 1.0924
Epoch 7/100
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4688 - loss: 1.0845

/Users/pmazolew/Documents/GitHub/AdvancedMachineLearning/pycaret-env/lib/python3.10/site-packages/keras/src/backend/tensorflow/nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5017 - loss: 1.0830 - val_accuracy: 0.4118 - val_loss: 1.0900
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4983 - loss: 1.0806 - val_accuracy: 0.4118 - val_loss: 1.0874
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4950 - loss: 1.0781 - val_accuracy: 0.4412 - val_loss: 1.0852
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5151 - loss: 1.0755 - val_accuracy: 0.4412 - val_loss: 1.0824
Epoch 11/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5084 - loss: 1.0731 - val_accuracy: 0.4412 - val_loss: 1.0800
Epoch 12/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5084 - loss: 1.0705 - val_accuracy: 0.4412 - val_loss: 1.0774
Epoch 13/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5017 - loss: 1.0681 - val_accuracy: 0.4412 - val_loss: 1.0751
Epoch 14/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5251 - loss: 1.0657 - val_accuracy: 0.4706 - val_loss: 1.0733
Epoch 1

In [175]:
model_logit_true = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_true.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits = False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_true.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)

scores = model_logit_true.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8060 - loss: 0.5414 - val_accuracy: 0.7941 - val_loss: 0.5380
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8027 - loss: 0.5297 - val_accuracy: 0.7941 - val_loss: 0.5263
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8027 - loss: 0.5218 - val_accuracy: 0.7941 - val_loss: 0.5192
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7993 - loss: 0.5147 - val_accuracy: 0.7941 - val_loss: 0.5142
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8027 - loss: 0.5089 - val_accuracy: 0.7941 - val_loss: 0.5077
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7993 - loss: 0.5034 - val_accuracy: 0.7941 - val_loss: 0.5009
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7993 - loss: 0.4975 - val_accuracy: 0.7941 - val_loss: 0.4962
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7993 - loss: 0.4925 - val_accuracy: 0.7941 - val_loss: 0.4912

In [176]:
model_logit_true.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([[7.33334780e-01, 2.66578019e-01, 8.71651719e-05],
       [2.12871701e-01, 7.86057174e-01, 1.07118790e-03],
       [7.83654273e-01, 2.16071978e-01, 2.73723505e-04],
       [2.84195325e-04, 1.79749858e-02, 9.81740832e-01],
       [7.57729232e-01, 2.41828606e-01, 4.42133605e-04],
       [4.81605202e-01, 5.05342007e-01, 1.30527308e-02],
       [7.67689109e-01, 2.31783330e-01, 5.27624157e-04],
       [2.15230375e-05, 5.33259427e-03, 9.94645953e-01],
       [8.45068280e-06, 3.71860573e-03, 9.96272981e-01],
       [4.98938898e-04, 2.15153154e-02, 9.77985680e-01],
       [4.25985634e-01, 5.52353263e-01, 2.16610264e-02],
       [9.15687919e-01, 8.43078643e-02, 4.11965812e-06],
       [6.98215604e-01, 3.01681906e-01, 1.02440448e-04],
       [8.30705941e-01, 1.69076040e-01, 2.17927081e-04],
       [4.96669412e-01, 5.03113806e-01, 2.16728542e-04],
       [8.16046894e-01, 1.83626786e-01, 3.26309470e-04],
       [8.23945701e-01, 1.75785482e-01, 2.68799282e-04],
       [2.88539479e-04, 1.46008

In [177]:
penguins['species']

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: object

In [178]:
penguins_y

array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,